## First regression model

Research question: How to forecast the
revenue of the next movie we create, based
on its characteristics only? 


## Load data and packages

In [209]:
import numpy as np
import pandas as pd   # We import Pandas!
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn import linear_model
import torch
import itertools

import pyro
import pyro.distributions as dist
from pyro.contrib.autoguide import AutoDiagonalNormal, AutoMultivariateNormal
from pyro.infer import MCMC, NUTS, HMC, SVI, Trace_ELBO
from pyro.optim import Adam, ClippedAdam

# fix random generator seed (for reproducibility of results)
np.random.seed(42)

# matplotlib options
palette = itertools.cycle(sns.color_palette())
plt.style.use('ggplot')
%matplotlib inline
plt.rcParams['figure.figsize'] = (16, 10)

In [210]:
df = pd.read_csv('../data/movies_with_genres_and_cast.csv')


C:\Users\samta\AppData\Local\Temp\ipykernel_25244\34911859.py:1: DtypeWarning: Columns (2) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('../data/movies_with_genres_and_cast.csv')


## Prepare data and matrix

In [211]:
# Keep only rows where revenue and budget are non-zero and not NaN
df = df[(df['revenue'].notna()) & (df['budget'].notna()) & (df['revenue'] != 0) & (df['budget'] != 0)]
print(f"Shape after removing zero and NaN revenue/budget: {df.shape}")

Shape after removing zero and NaN revenue/budget: (7428, 28)


In [262]:
X_time = np.concatenate([pd.get_dummies(df[x]).astype(float) for x in ["dow", "week"]], axis=1)
print(X_time.shape)


(7428, 60)


In [263]:
X_movie = df[["budget", "genre_drama" , "genre_comedy" , "genre_thriller",
 "genre_romance", "genre_action", "genre_horror", "genre_crime",
 "genre_documentary", "genre_adventure", "genre_science_fiction"]].copy()

# Convert all columns to numeric
for i in X_movie.columns:
    X_movie[i] = pd.to_numeric(X_movie[i], errors='coerce')
X_movie = X_movie.values

print(X_movie.shape)
X_movie[1]

# Note: maybe have 1/n for n the number of genre we consider only, not all genres

(7428, 11)


array([6.5e+07, 0.0e+00, 0.0e+00, 0.0e+00, 0.0e+00, 0.0e+00, 0.0e+00,
       0.0e+00, 0.0e+00, 3.3e-01, 0.0e+00])

In [264]:
# create matrix with time and movie features
X = np.concatenate([X_time, X_movie], axis=1)
print(X.shape)


(7428, 71)


In [265]:
X_mean = X.mean(axis=0)
X_std = X.std(axis=0)
X = (X - X_mean) / X_std
print(f"Number of NaN in X: {np.isnan(X).sum()}")

Number of NaN in X: 0


In [266]:
# prepare vector of revenue
y = df["revenue"].values

# standardize revenue
y_mean = y.mean()
y_std = y.std()
y = (y - y_mean) / y_std
print(y.shape)
print(f"Number of NaN in y: {np.isnan(y).sum()}")


(7428,)
Number of NaN in y: 0


In [267]:
train_perc = 0.66 # percentage of training data
split_point = int(train_perc*len(y))
perm = np.random.permutation(len(y))
ix_train = perm[:split_point]
ix_test = perm[split_point:]
X_time_train = X_time[ix_train,:]
X_time_test = X_time[ix_test,:]
X_train = X[ix_train,:]
X_test = X[ix_test,:]
y_train = y[ix_train]
y_test = y[ix_test]
print("num train: %d" % len(y_train))
print("num test: %d" % len(y_test))

num train: 4902
num test: 2526


## Test and error mesurement

In [268]:
def compute_error(trues, predicted):
    corr = np.corrcoef(predicted, trues)[0,1]
    mae = np.mean(np.abs(predicted - trues))
    rae = np.sum(np.abs(predicted - trues)) / np.sum(np.abs(trues - np.mean(trues)))
    rmse = np.sqrt(np.mean((predicted - trues)**2))
    r2 = max(0, 1 - np.sum((trues-predicted)**2) / np.sum((trues - np.mean(trues))**2))
    return corr, mae, rae, rmse, r2

In [269]:
#regr = linear_model.LinearRegression()
regr = linear_model.Ridge()
regr.fit(X_train, y_train)
y_hat = regr.predict(X_test)

# Convert back to the original scale
preds = y_hat * y_std + y_mean
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))

CorrCoef: 0.747
MAE: 50272725.906
RMSE: 100452728.899
R2: 0.557


## MODEL

In [412]:
y_mean = df["revenue"].mean()
print(f"Mean revenue: {y_mean}")
X.shape[1]

Mean revenue: 68649475.2023425


71

In [436]:
def model(X, obs=None):
    alpha = pyro.sample("alpha", dist.Normal(y_mean, y_mean*0,5))                   # Prior for the bias/intercept
    beta  = pyro.sample("beta", dist.Normal(torch.zeros(X.shape[1]), 
                                            torch.ones(X.shape[1])).to_event())    # Priors for the regression coeffcients
    sigma = pyro.sample("sigma", dist.HalfCauchy(0.5))                   # Prior for the variance
    
    with pyro.plate("data"):
        y = pyro.sample("y", dist.Normal(alpha + X.matmul(beta), sigma), obs=obs)
        
    return y

## SVI training

In [437]:
# Prepare data for Pyro model
X_train_small = torch.tensor(X_train[:7000,:]).float()
y_train_small = torch.tensor(y_train[:7000]).float()
print(X_train_small)

tensor([[-0.2129, -0.2596,  2.2307,  ..., -0.1652, -0.3896, -0.3058],
        [-0.2129, -0.2596, -0.4483,  ..., -0.1652, -0.3896, -0.3058],
        [-0.2129, -0.2596, -0.4483,  ..., -0.1652, -0.3896, -0.3058],
        ...,
        [-0.2129, -0.2596,  2.2307,  ..., -0.1652, -0.3896,  2.7437],
        [-0.2129, -0.2596, -0.4483,  ..., -0.1652, -0.3896, -0.3058],
        [-0.2129, -0.2596, -0.4483,  ..., -0.1652, -0.3896, -0.3058]])


In [438]:
# Prepare data for Pyro model
X_train_torch = torch.tensor(X_train).float()
y_train_torch = torch.tensor(y_train).float()

In [439]:
# Define guide function
guide = AutoMultivariateNormal(model)

# Reset parameter values
pyro.clear_param_store()

In [440]:
# Define the number of optimization steps
n_steps = 4000

# Setup the optimizer
adam_params = {"lr": 0.001} # learning rate (lr) of optimizer
optimizer = ClippedAdam(adam_params)

# Setup the inference algorithm
elbo = Trace_ELBO(num_particles=1)
svi = SVI(model, guide, optimizer, loss=elbo)

In [ ]:
# Do gradient steps
alpha = pyro.sample("alpha", dist.Normal(y_mean, y_mean*0,5))


ValueError: Expected parameter scale (Tensor of shape ()) of distribution Normal(loc: 68649472.0, scale: 0.0) to satisfy the constraint GreaterThan(lower_bound=0.0), but found invalid values:
0.0
Trace Shapes:
 Param Sites:
Sample Sites:
Trace Shapes:
 Param Sites:
Sample Sites:

In [ ]:
from pyro.infer import Predictive

predictive = Predictive(model, guide=guide, num_samples=1000,
                        return_sites=("alpha", "beta", "sigma"))
samples = predictive(X_train_torch, y_train_torch)

In [ ]:
alpha_samples = samples["alpha"].detach().numpy()
beta_samples = samples["beta"].detach().numpy()
y_hat = np.mean(alpha_samples.T + np.dot(X_test, beta_samples[:,0].T), axis=1)

# convert back to the original scale
preds = y_hat * y_std + y_mean
y_true = y_test * y_std + y_mean

corr, mae, rae, rmse, r2 = compute_error(y_true, preds)
print("CorrCoef: %.3f\nMAE: %.3f\nRMSE: %.3f\nR2: %.3f" % (corr, mae, rmse, r2))


CorrCoef: 0.135
MAE: 14283618378590932.000
RMSE: 14283618378591754.000
R2: 0.000
